In [1]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [2]:
import torch
from torchvision import transforms
def transform_to_tensor_with_resize(images,labels):
  transform = transforms.Compose([transforms.Resize((224, 224)),transforms.ToTensor()])
  image_tensors = torch.stack([transform(image) for image in images])

  labels_tensor = torch.tensor(labels)

  print("Image tensor shape:", image_tensors.shape)
  print("Labels shape:", labels_tensor.shape)
  return image_tensors,labels_tensor

In [3]:
def model_train(model, image_tensors,labels_tensor):
    
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
    
    num_epochs = 80

    for epoch in range(num_epochs):

      # Forward pass
      outputs = model(image_tensors)

      # Calculate loss
      loss = loss_function(outputs, labels_tensor)

      # Clear old gradients
      optimizer.zero_grad()

      # Backpropagation
      loss.backward()

      # Update weights
      optimizer.step()

      print("Epoch:", epoch + 1,"Loss:", loss.item())
    with torch.no_grad():
      outputs = model(image_tensors)
      predictions = torch.argmax(outputs, dim=1)

    print("Actual labels:    ", labels_tensor.tolist())
    print("Predicted labels: ", predictions.tolist())

In [4]:
import zipfile
from PIL import Image

def load_one_frame_per_video(zip_path, real_count, fake_count):

    images = []
    labels = []

    with zipfile.ZipFile(zip_path, "r") as z:

        # ---------- REAL ----------
        real_folders = sorted(set(
            path.split("/")[1]
            for path in z.namelist()
            if path.startswith("real/") and path.endswith(".png")
        ))

        for folder in real_folders[:real_count]:

            # Find first PNG inside this video folder
            files = [
                path for path in z.namelist()
                if path.startswith(f"real/{folder}/")
                and path.endswith(".png")
            ]

            if len(files) == 0:
                continue

            image_path = files[0]

            with z.open(image_path) as f:
                image = Image.open(f).convert("RGB")
                images.append(image.copy())

            labels.append(0)   # 0 = Real


        # ---------- DEEPFAKE ----------
        fake_folders = sorted(set(
            path.split("/")[2]
            for path in z.namelist()
            if path.startswith("fake/Deepfakes/")
            and path.endswith(".png")
        ))

        for folder in fake_folders[:fake_count]:

            files = [
                path for path in z.namelist()
                if path.startswith(f"fake/Deepfakes/{folder}/")
                and path.endswith(".png")
            ]

            if len(files) == 0:
                continue

            image_path = files[0]

            with z.open(image_path) as f:
                image = Image.open(f).convert("RGB")
                images.append(image.copy())

            labels.append(1)   # 1 = Deepfake

    return images, labels

In [5]:
images, labels = load_one_frame_per_video("FF++.zip",200,200)
model= SimpleCNN()
image_tensors,labels_tensor=transform_to_tensor_with_resize(images,labels)
model_train(model,image_tensors,labels_tensor)

Image tensor shape: torch.Size([400, 3, 224, 224])
Labels shape: torch.Size([400])
Epoch: 1 Loss: 0.697928249835968
Epoch: 2 Loss: 3.889866352081299
Epoch: 3 Loss: 0.6909680962562561
Epoch: 4 Loss: 1.5284062623977661
Epoch: 5 Loss: 1.1958379745483398
Epoch: 6 Loss: 0.8817122578620911
Epoch: 7 Loss: 0.7631134986877441
Epoch: 8 Loss: 0.7156184315681458
Epoch: 9 Loss: 0.6962310671806335
Epoch: 10 Loss: 0.6930318474769592
Epoch: 11 Loss: 0.6969760656356812
Epoch: 12 Loss: 0.7025710344314575
Epoch: 13 Loss: 0.7061142921447754
Epoch: 14 Loss: 0.6995446681976318
Epoch: 15 Loss: 0.6949760913848877
Epoch: 16 Loss: 0.6927743554115295
Epoch: 17 Loss: 0.6932926177978516
Epoch: 18 Loss: 0.6940836906433105
Epoch: 19 Loss: 0.693701982498169
Epoch: 20 Loss: 0.6925202012062073
Epoch: 21 Loss: 0.6910151839256287
Epoch: 22 Loss: 0.689948320388794
Epoch: 23 Loss: 0.6900671124458313
Epoch: 24 Loss: 0.6896242499351501
Epoch: 25 Loss: 0.6882683038711548
Epoch: 26 Loss: 0.6872410774230957
Epoch: 27 Loss: 0.68

In [23]:
def test_set(folder_name):
 test_images=[]
 test_labels=[]

 data_folder=zipfile.Path("CelebDF.zip")
    
 # Loading fake images
 FAKE_DIR =data_folder/f'Celeb_V2/{folder_name}/fake'
 image_files =[x for x in FAKE_DIR.iterdir() if x.name.endswith(".jpg")][:100]
 for image_path in image_files:
    with image_path.open("rb") as f:
     image = Image.open(f).convert("RGB")
     test_images.append(image)
 for i in range(100):
    test_labels.append(1)

 # Loading real images
 REAL_DIR =data_folder/f'Celeb_V2/{folder_name}/real'
 image_files = [x for x in REAL_DIR.iterdir() if x.name.endswith(".jpg")][:100]
 for image_path in image_files:
    with image_path.open("rb") as f:
      image=Image.open(f).convert("RGB")  
      test_images.append(image)
 for i in range(100):
    test_labels.append(0)
 print("Images:", len(test_images))
 print("Labels:", len(test_labels))
 return test_images,test_labels


In [24]:
def model_test(model, image_tensors,labels_tensor):
    loss_function = nn.CrossEntropyLoss()

    # Forward Pass
    with torch.no_grad():
      
      outputs = model(image_tensors)
      # Calculate loss
      loss = loss_function(outputs, labels_tensor)
      predictions = torch.argmax(outputs, dim=1)

       # Probability/score that an image is fake
      fake_scores = torch.softmax(outputs, dim=1)[:, 1]

    print("Test Loss:", loss.item())
    print("Actual labels:    ", labels_tensor.tolist())
    print("Predicted labels: ", predictions.tolist())
    return predictions,fake_scores

In [25]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.metrics import roc_curve
import numpy as np


def calculate_metrics(labels, predictions, scores):

    accuracy = accuracy_score(labels, predictions)

    precision = precision_score(labels, predictions, zero_division=0)

    recall = recall_score(labels, predictions, zero_division=0)

    f1 = f1_score(labels, predictions, zero_division=0)

    auc = roc_auc_score(labels, scores)

    ap = average_precision_score(labels, scores)

    pauc = roc_auc_score(
        labels,
        scores,
        max_fpr=0.10
    )

    fpr, tpr, thresholds = roc_curve(labels, scores)
    fnr = 1 - tpr

    index = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[index] + fnr[index]) / 2

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1       :", f1)
    print("AUC      :", auc)
    print("AP       :", ap)
    print("pAUC     :", pauc)
    print("EER      :", eer)

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "AUC": auc,
        "AP": ap,
        "pAUC": pauc,
        "EER": eer
    }

In [26]:
print("----------------------Testing the CNN Model on unseen Celeb-DF V2 dataset--------------------------\n")
test_images,test_labels=test_set('Test')
test_image_tensors,test_labels_tensor= transform_to_tensor_with_resize(test_images,test_labels)
predictions, fake_scores=model_test(model,test_image_tensors,test_labels_tensor)
metrics = calculate_metrics(test_labels_tensor.numpy(),predictions.numpy(),fake_scores.numpy())

----------------------Testing the CNN Model on unseen Celeb-DF V2 dataset--------------------------

Images: 200
Labels: 200
Image tensor shape: torch.Size([200, 3, 224, 224])
Labels shape: torch.Size([200])
Test Loss: 0.7892263531684875
Actual labels:     [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Predicted labels:  [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0,

In [34]:
def test_load_one_frame_per_video(zip_path,folder_name, real_count, fake_count):

    images = []
    labels = []

    with zipfile.ZipFile(zip_path, "r") as z:

        # ---------- REAL ----------
        real_folders = sorted(set(
            path.split("/")[1]
            for path in z.namelist()
            if path.startswith("real/") and path.endswith(".png")
        ))

        for folder in real_folders[200:(real_count+200)]:

            # Find first PNG inside this video folder
            files = sorted([
                path for path in z.namelist()
                if path.startswith(f"real/{folder}/")
                and path.endswith(".png")
            ])

            if len(files)==0:
                continue

            image_path = files[0]

            with z.open(image_path) as f:
                image = Image.open(f).convert("RGB")
                images.append(image.copy())

            labels.append(0)   # 0 = Real


        # ---------- DEEPFAKE ----------
        fake_folders = sorted(set(
            path.split("/")[2]
            for path in z.namelist()
            if path.startswith(f"fake/{folder_name}/")
            and path.endswith(".png")
        ))

        for folder in fake_folders[:fake_count]:

            files = sorted([
                path for path in z.namelist()
                if path.startswith(f"fake/{folder_name}/{folder}/")
                and path.endswith(".png")
            ])

            if len(files)==0:
                continue

            image_path = files[0]

            with z.open(image_path) as f:
                image = Image.open(f).convert("RGB")
                images.append(image.copy())

            labels.append(1)   # 1=manipulated image

    return images, labels

In [35]:
print("------------------------Testing on unseen Face2Face dataset from FaceForensics++-----------------------\n") 
images_face2face, labels_face2face= test_load_one_frame_per_video("FF++.zip","Face2Face",100,100)
image_tensors_face2face,labels_tensor_face2face= transform_to_tensor_with_resize(images_face2face, labels_face2face)
predictions, fake_scores=model_test(model,image_tensors_face2face,labels_tensor_face2face)
metrics = calculate_metrics(labels_tensor_face2face.numpy(),predictions.numpy(),fake_scores.numpy())

------------------------Testing on unseen Face2Face dataset from FaceForensics++-----------------------

Image tensor shape: torch.Size([200, 3, 224, 224])
Labels shape: torch.Size([200])
Test Loss: 0.7601408362388611
Actual labels:     [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Predicted labels:  [1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0

In [36]:
print("------------------------Testing on unseen FaceShifter dataset from FaceForensics++-----------------------\n") 
images_faceshift, labels_faceshift= test_load_one_frame_per_video("FF++.zip","FaceShifter",100,100)
image_tensors_faceshift,labels_tensor_faceshift= transform_to_tensor_with_resize(images_faceshift, labels_faceshift)
predictions, fake_scores=model_test(model,image_tensors_faceshift,labels_tensor_faceshift)
metrics = calculate_metrics(labels_tensor_faceshift.numpy(),predictions.numpy(),fake_scores.numpy())

------------------------Testing on unseen FaceShifter dataset from FaceForensics++-----------------------

Image tensor shape: torch.Size([200, 3, 224, 224])
Labels shape: torch.Size([200])
Test Loss: 0.7526611089706421
Actual labels:     [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Predicted labels:  [1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1,

In [37]:
print("-----------------------------Testing on unseen FaceSwap dataset from FaceForensics++-------------------------\n") 
images_faceswap, labels_faceswap= test_load_one_frame_per_video("FF++.zip","FaceSwap",100,100)
image_tensors_faceswap,labels_tensor_faceswap= transform_to_tensor_with_resize(images_faceswap, labels_faceswap)
predictions, fake_scores=model_test(model,image_tensors_faceswap,labels_tensor_faceswap)
metrics = calculate_metrics(labels_tensor_faceswap.numpy(),predictions.numpy(),fake_scores.numpy())

-----------------------------Testing on unseen FaceSwap dataset from FaceForensics++-------------------------

Image tensor shape: torch.Size([200, 3, 224, 224])
Labels shape: torch.Size([200])
Test Loss: 0.7471820712089539
Actual labels:     [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Predicted labels:  [1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1